## 0. Init libraries en helper functions

In [79]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [80]:
import pandas as pd
import re
import numpy as np

# from arcgis.gis import GIS
# from arcgis.apps import survey123

import sqlite3
from datetime import datetime

In [81]:
import os
import sys

# Get the absolute path of the folder above the notebook
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))

if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from src import utils

In [82]:
pd.options.display.max_columns = None

### 0.1 Inladen gegevens

Gebruikt nu nog exports via LSVI-package (R). Testen of export uit sqlite db kan gemaakt worden.

In [83]:
df_vereisten = pd.read_excel('../input/LSVI_packageInvoervereisten_uitdb_2026-06-08_aanvullingenLSVI_app.xlsx', sheet_name='LSVI_packageInvoervereisten_uit')
# We should only use vereiesten v3
df_vereisten = df_vereisten[df_vereisten['Versie'] == 'Versie 3']

In [84]:
df_vereisten = pd.read_excel('../input/LSVI_packageInvoervereisten_uitdb_2026-06-08_aanvullingenLSVI_app.xlsx', sheet_name='LSVI_packageInvoervereisten_uit')
# We should only use vereiesten v3
df_vereisten = df_vereisten[df_vereisten['Versie'] == 'Versie 3']
print(df_vereisten.shape)
# Maak unieke ID aan voor vragen adhv voorwaarde id + habitattype
df_vereisten['vraag_id'] = "vraag_" + df_vereisten['VoorwaardeID'].astype(str) + "_" + (df_vereisten['Habitatsubtype'].astype(str).apply(utils.clean_name))
# Make sure vereisten ID (VoorwaardeID) is uniek
df_vereisten.drop_duplicates(subset=['vraag_id'], inplace=True)
print(df_vereisten.shape)

# Type vraag/antwoord is combinatie van kolom Schaal en AnalyseVariabele
# Coalesce kolom Schaal (prioritair) en AnalyseVariabele (fallback)
df_vereisten['schaal_type'] = df_vereisten.apply(lambda row: row['Schaal'] if pd.notnull(row['Schaal']) else row['AnalyseVariabele'], axis=1)

if len(df_vereisten[df_vereisten['schaal_type'].isnull()]) > 0:
    print("Er zijn vereisten zonder type vraag/antwoord. Controleer de invoervereisten!")
    print(df_vereisten[df_vereisten['schaal_type'].isnull()])


print("Aantal unieke schaal in de vereisten:", df_vereisten['schaal_type'].unique())

df_soorten = pd.read_csv('../input/invoervereistenUitTeWerkenSoortenlijst-UTF8.csv', sep=';')
df_schalen = pd.read_csv('../input/survey123_schalen.csv', sep=';') # Het bestand van de vorige stap!

(710, 39)
(710, 40)
Aantal unieke schaal in de vereisten: <ArrowStringArray>
[            'LSVI',           'meting',     'maxBedekking',
        'bedekking',    'meting_opp_m2', 'meting_bedekking',
      'meting_perc',    'meting_opp_ha',         'scoresom',
         'meting_m']
Length: 10, dtype: str


In [85]:
# Groepen voor matrixvragen
df_groepen = pd.read_excel('../input/LSVI_packageInvoervereisten_uitdb_2026-06-08_aanvullingenLSVI_app.xlsx', sheet_name='Groepen')
# 2. Opschonen: Verwijder eventuele onzichtbare spaties aan de randen van de tekst
df_groepen['Name'] = df_groepen['Name'].astype(str).str.strip()
df_groepen['Value'] = df_groepen['Value'].astype(str).str.strip()

# 3. DE TRUC: Groepeer op 'Name', maak een lijst van 'Value', en zet om naar een dictionary
groepen_mapping = df_groepen.groupby('Name')['Value'].apply(list).to_dict()

In [86]:
# df_vereisten[['AnalyseVariabele','Eenheid']].drop_duplicates()

**Komt reeds uit SQLite**

In [87]:
df_habitattypes = pd.read_sql_table('Habitattype', 'sqlite:///../input/LSVIHabitatTypes.sqlite')

In [88]:
# df_habitattypes[~df_habitattypes['Code'].str.startswith('rbb')].HabitatgroepId.unique()
# df_habitattypes[~df_habitattypes['Code'].str.startswith('rbb')].Code.unique()

**BWK-karteringseenheden: betere bron nog te gebruiken**

In [89]:
df_bwk_kart_eenh = pd.read_csv('../input/bwk_karteringseenheden.csv', sep=';')

In [90]:
df_vereisten[df_vereisten['vraag_id'] == '6410_mo']  #vraag_rbbsm_verruiging_verruiging_sublijst_2
df_vereisten.head()
# df_habitattypes[df_habitattypes['Code'] == '6410']
df_vereisten.Habitattype.unique()
df_soorten[df_soorten['TaxongroepId'] == 1]

,TaxongroepId,TaxonsubgroepId,Omschrijving,Id,NbnTaxonVersionKey,WetNaam,NedNaam,TaxonType,WetNaamKort
0,1,1,aanwezigheid Sierlijke Vetmuur_Zeevetmuur,444,INBSYS0000004166,Sagina maritima G. Don,Zeevetmuur,Species,Sagina maritima
1,1,1,aanwezigheid Sierlijke Vetmuur_Zeevetmuur,1529,NBNSYS0000003051,Sagina nodosa (L.) Fenzl,Sierlijke vetmuur,Species,Sagina nodosa


In [91]:
# df_vereisten = df_vereisten[df_vereisten['Habitattype'].astype(str).isin(['9110','9120'])]

In [92]:
df_vereisten.shape

(710, 41)

## 1. Choices

In [93]:
print("Choices tabblad opbouwen...")
choices_list = []

# A. Schalen toevoegen
for _, row in df_schalen.iterrows():
    choices_list.append(row.to_dict())

# B. Dynamische Habitattype lijst genereren (Voor de eerste vraag)
unieke_habitats = df_vereisten['Habitattype'].dropna().unique()
for hab in unieke_habitats:
    choices_list.append({
        "list_name": "lijst_habitats",
        "name": utils.clean_name(hab),
        "label": str(hab).upper()
    })

# C. Dynamische Soortenlijsten genereren (Groepeer per TaxongroepId)
for _, soort in df_soorten.iterrows():
    if pd.notna(soort['TaxongroepId']):
        tax_id = int(soort['TaxongroepId'])
        choices_list.append({
            "list_name": f"taxa_{tax_id}",
            "name": utils.clean_name(soort['WetNaamKort']),
            "label": f"{soort['NedNaam']} ({soort['WetNaamKort']})" if pd.notna(soort['NedNaam']) else soort['WetNaamKort']
        })

# Dynamische subhapitattypes toevoegen (Voor de BWK-vragen)
unieke_subhabitats = df_vereisten['Habitatsubtype'].dropna().unique()
for hab in unieke_subhabitats:
    choices_list.append({
        "list_name": "lijst_subhabitats",
        "name": utils.clean_name(hab),
        "label": str(hab).upper()
    })

# BWK Karteringseenheden toevoegen
for _, row in df_bwk_kart_eenh.iterrows():
    choices_list.append({
        "list_name": "lijst_bwk_karteringseenheden",
        "name": utils.clean_name(row["karteringseenheid"]),
        "label": str(row['karteringseenheid'])
    })

# BWK Verhoudingen toevoegen
bwk_verhoudingen = ['1 op 2', '2 op 3', '3 op 4', '4 op 5']
for verhouding in bwk_verhoudingen:
    choices_list.append({
        "list_name": "lijst_bwk_verhoudingen",
        "name": utils.clean_name(verhouding),
        "label": verhouding
    })
    
df_choices_final = pd.DataFrame(choices_list)

# Zorg dat de talen overeenkomen in survey en choices!
if 'label' in df_choices_final.columns:
    df_choices_final.rename(columns={'label': 'label::nl'}, inplace=True)



Choices tabblad opbouwen...


In [94]:
df_choices_final[df_choices_final['list_name'] == 'lijst_subhabitats'].dtypes

list_name    str
name         str
label::nl    str
dtype: object

In [95]:
# Add settings dataframe
settings_df = pd.DataFrame([{
    "form_id": "lsvi_app_test",
    "form_title": "LSVI App Test",
    "style": "pages",   # <-- DIT ACTIVEERT HET GRID SYSTEEM IN DE APP
    "default_language": "nl" 
}])

**Waarschuwing: veel duplicate soorten in verschillende lijsten --> is dit normaal?**

## 2. Survey

### 2.1 Algemene info

In [96]:
print("Survey tabblad opbouwen...")
survey_list = []

# Unieke ID: Wordt op de achtergrond gegenereerd (gebruiker ziet dit niet)
survey_list.append({
    "type": "calculate", "name": "collectie_id", "label": "", 
    "calculation": "uuid()", "relevant": "", "appearance": "", "default": ""
})

# Datum & Uur: Automatisch ingevuld
survey_list.append({
    "type": "date", "name": "datum", "label": "Datum", 
    "default": "today()", "relevant": "", "appearance": "", "calculation": ""
})

survey_list.append({
    "type": "time", "name": "uur", "label": "Uur", 
    "default": "now()", "relevant": "", "appearance": "", "calculation": ""
})

Survey tabblad opbouwen...


### 2.2 BWK-vragen

**In finale versie kunnen velden op appearance: hidden gezet worden.**

In [97]:
# Toon BWK ID over hele breedte
survey_list.append({
    "type": "integer", 
    "name": "bwk_id", 
    "label": "BWK ID", 
    "default": "", 
    "relevant": "", 
    "appearance": "",  # Neemt 5/5 kolommen in, dus over hele rij
    "calculation": "", 
    "readonly": "yes"
})

# Maak grid aan voor hab and phab velden weer te geven
survey_list.append({
    "type": "begin group", 
    "name": "grp_fieldmaps_data", 
    "label": "Gegevens uit Field Maps (Controle)", 
    "relevant": "", 
    "appearance": "w5 fixed-grid"  # <-- Dit activeert het grid-systeem!
})



# Toon HAB en PHAB in grid
for i in range(1, 6):
    # Top row: Habitat Code (Breedte = w1)
    survey_list.append({
        "type": "text", 
        "name": f"hab{i}", 
        "label": f"Hab. {i}", 
        "default": "", 
        "relevant": "", 
        "appearance": "w1",  # <-- Neem 1/5 kolommen in
        "calculation": "", 
        "readonly": "no"
    })

for i in range(1, 6):
    # Bottom row: Percentage (Breedte = w1)
    survey_list.append({
        "type": "integer", 
        "name": f"phab{i}", 
        "label": f"% Hab. {i}", 
        "default": "", 
        "relevant": "", 
        "appearance": "w1",  # <-- Neemt de andere helft van de ruimte in
        "calculation": "", 
        "readonly": "yes"
    })

# 4. Sluit de grid groep netjes af
survey_list.append({
    "type": "end group", "name": "", "label": "", "relevant": "", "appearance": ""
})

In [98]:
df_survey_final = pd.DataFrame(survey_list)

### 2.3 Vragen per habitattype

In [99]:
# df_vereisten.AnalyseVariabele.unique()
df_soorten[df_soorten['TaxongroepId'] == 589]
# df_vereisten[df_vereisten['Habitattype'] == 'rbbsg'][['Habitattype','Indicator','VoorwaardeID','Voorwaarde','AnalyseVariabele', 'Eenheid', 'TypeVariabele']]
# df_vereisten[df_vereisten['VoorwaardeID'] == 2198]
# pd.DataFrame(survey_list).head()

,TaxongroepId,TaxonsubgroepId,Omschrijving,Id,NbnTaxonVersionKey,WetNaam,NedNaam,TaxonType,WetNaamKort
9594,589,589,verruiging sublijst 3,1865,NBNSYS0000003807,Urtica dioica L.,Grote brandnetel,Species,Urtica dioica
9595,589,589,verruiging sublijst 3,2060,NBNSYS0000004257,Glechoma hederacea L.,Hondsdraf,Species,Glechoma hederacea
9596,589,589,verruiging sublijst 3,2101,NBNSYS0000004319,Galium aparine L.,Kleefkruid,Species,Galium aparine
9597,589,589,verruiging sublijst 3,2105,NBNSYS0000004324,Sambucus nigra L.,Gewone vlier,Species,Sambucus nigra


In [100]:
# Trigger vraag
survey_list.append({
    "type": "select_one Ja_Nee", "name": "lsvi_opstellen", "label": "LSVI Opstellen?", 
    "relevant": "", "appearance": "horizontal", "default": "", "calculation": ""
})

# Groepeer alle LSVI vragen zodat we de 'relevant' logica maar 1 keer hoeven te typen
survey_list.append({
    "type": "begin group", "name": "grp_lsvi", "label": "LSVI Gegevens", 
    "relevant": "${lsvi_opstellen} = 'ja'", # Zichtbaar als vorige vraag 'ja' is
    "appearance": "field-list", "default": "", "calculation": ""
})

# Eerste hoofdvraag: Welk habitattype?
survey_list.append({
    "type": "select_multiple lijst_subhabitats",
    "name": "habitat_keuze",
    "label": "Welk habitat(sub)type wil je inventariseren?",
    "relevant": "",  # Altijd zichtbaar
    "hint": "Eenvoudige tekst om feature te testen.",
    "guidance_hint": "Eenvoudige tekst om feature te testen.",
    "appearance": "horizontal", #blank defaults to radio buttons instead of "minimal autocomplete",
    "choice_filter": "string(name) = string(${hab1}) or string(name) = string(${hab2}) or string(name) = string(${hab3}) or string(name) = string(${hab4}) or string(name) = string(${hab5})" # This makes sure we only get to choose habitats that were mapped in BWK field app for this polygon.
})



# 4.2. Loop door de unieke habitattypes (Creëer "Pages" / Groups)
for hab in unieke_subhabitats:
    hab_clean = utils.clean_name(hab)

    # Vragen per habitattype
    # Begin de groep voor dit specifieke habitattype. 
    # De "relevant" zorgt ervoor dat enkel de habitattypes uit BWK-kaart worden weergegeven in app.
    survey_list.append({
        "type": "begin group",
        "name": f"grp_habitat_{hab_clean}",
        "label": f"Habitat {hab_clean.upper()}",
        "hint": utils.get_habitat_hint(hab),
        # "guidance_hint": get_habitat_hint(hab),   # Dynamische hint genereren op basis van de habitattype code
        "relevant": f"selected(string(${{habitat_keuze}}), '{hab_clean}')",   # De groep erft de relevantie van het repeat blok. Dit mag leeg zijn als we repeats gebruiken.
        "appearance": "w1 compact field-list" # Zorgt dat het als 1 pagina toont in de app
    })

    # Filter de vereisten voor dít specifieke habitattype
    df_hab_vereisten = df_vereisten[df_vereisten['Habitatsubtype'] == hab]
    
    # 4.3. Genereer de vragen binnen dit habitattype
    for idx, row in df_hab_vereisten.iterrows():
        vraag_naam = f"{row['vraag_id']}"

        # Type_vraag heeft 3 mogelijkheden: Orig (normaal), Matrixvraag of 'niet nodig in app':
        if row['Type_vraag'].lower() == 'orig':
            # Do usual processing
            answer_type, vraag_appearance = utils.get_question_settings(row)

            # Vraag toevoegen
            # Label van de vraag is combinatie van Voorwaarde + Indicator + Beoordeling(en eventueel Eenheid)
            # Add soortenlijst als hint 
            survey_list.append({
                "type": answer_type,
                "name": vraag_naam,
                "label": utils.get_question_label(row), # De vraag die de gebruiker ziet
                "relevant": "",
                "appearance": vraag_appearance
            })
        
        elif row['Type_vraag'].lower() == 'matrixvraag':
            # Moet iets doen voor alle groepen? Zie groep kolom? 
            
            # Start subgroep voor matrix met grid layout
            survey_list.append({
                "type": "begin group",
                "name": f"mat_grp_{vraag_naam}",
                "label": utils.get_question_label(row), # Genereert jouw mooie HTML label
                "hint": "Scoor elk van de onderstaande onderdelen volgens de LSVI-schaal.",
                "relevant": "",
                "appearance": "w2 grid-layout" # Activeert het 2-koloms rastersysteem
            })

            # Welke groep moeten we bevragen in matrix? Sleutelsoorten of andere categorie? 
            groep_naam = str(row['Groepen']).strip().lower()
            items_te_scoren = []
            
            if 'sleutelsoorten' in groep_naam:
                # CASE A: Haal de specifieke soorten op uit df_soorten op basis van TaxongroepId
                tax_id = row['TaxongroepId']
                if pd.notna(tax_id):
                    df_sub_soorten = df_soorten[df_soorten['TaxongroepId'] == int(tax_id)]
                    # Pak 'NedNaam', tenzij NaN, dan pak 'WetNaam'
                    items_te_scoren = df_sub_soorten['NedNaam'].fillna(df_sub_soorten['WetNaam']).tolist()
            else:
                # CASE B: Haal de vaste categorieën op uit het tabblad 'Groepen' (de dictionary)
                raw_groep = str(row['Groepen']).strip()
                items_te_scoren = groepen_mapping.get(raw_groep, [])

            # Veiligheidscheck voor als er niets gevonden is
            if not items_te_scoren:
                print(f"Waarschuwing: Geen matrix items gevonden voor groep '{row['Groepen']}' bij vraag {vraag_naam}")

            # Genereer de rijen (Text + Dropdown paren) binnen het grid
            for item in items_te_scoren:
                # We saneren de tekst handmatig naar kleine letters zonder vreemde tekens
                item_clean = re.sub(r'[^a-z0-9]', '', str(item).lower().strip())
                voorwaarde_clean = re.sub(r'[^a-z0-9]', '', str(row['VoorwaardeID']).lower().strip())
                
                # SLIMME TRUC: Omdat jouw utils.clean_name strikt maximaal 2 delen (Deel1_Deel2) 
                # toestaat, bouwen we de veldnaam op als "v{VoorwaardeID}_{item}". 
                # Dit levert exact 2 delen op (bijv: v317_pionierstadium) wat perfect uniek is!
                uniek_veld_name = f"v{voorwaarde_clean}_{item_clean}"
                uniek_veld_name = uniek_veld_name[0:27] # Beperkt tot 32 tekens

                # KOLOM 1 (Links): De naam van de soort of het stadium (w6 = 50% breedte)
                survey_list.append({
                    "type": "note",
                    "name": f"lbl_{uniek_veld_name}", # Beperkt tot 32 tekens
                    "label": f"{item.capitalize()}",
                    "relevant": "",
                    "appearance": "w1" 
                })
                
                # KOLOM 2 (Rechts): De LSVI Keuzelijst Dropdown (w6 = 50% breedte)
                survey_list.append({
                    "type": "select_one LSVI",
                    "name": uniek_veld_name,
                    "label": "Score:", # Spatie verbergt het label visueel, maar voorkomt Connect fouten
                    "relevant": "",
                    "appearance": "minimal w1" # 'minimal' maakt er een dropdown van
                })

            # 4. Sluit de matrix sub-groep netjes af
            survey_list.append({
                "type": "end group", "name": "", "label": "", "relevant": "", "appearance": ""
            })


        else:
            # Skip question
            continue
        
        

        # Onder vraag wil je soms de soortenlijst weergeven.
        # Doen dit niet als hint of guidance_hint in de vraag, want niet inklapbaar of te weinig ruimte.
        # We maken een aparte groep aan (conditioneel) met soorten indien nodig.
        # Genereer de soortenlijst HTML
        if row["Soortenlijst weergeven in vraag"] == 1:
            html_soorten = utils.get_species_hint(row['TaxongroepId'], df_soorten)

            # Check of er een soortenlijst is om te tonen
            if pd.notna(html_soorten) and str(html_soorten).strip() != "":
                # Start de in- en uitklapbare groep
                survey_list.append({
                    "type": "begin group",
                    "name": f"grp_lijst_{vraag_naam}",
                    "label": "🔽 Bekijk soortenlijst", # Dit is de tekst op de klikbare balk
                    "hint": np.nan,
                    "guidance_hint": np.nan,
                    "relevant": "",
                    "appearance": "compact" # <--- Dit commando maakt hem standaard ingeklapt!
                })

                # Voeg de 'note' toe met jouw HTML-geformatteerde soortenlijst
                survey_list.append({
                    "type": "note",
                    "name": f"not_{vraag_naam}",
                    "label": html_soorten, # We stoppen jouw HTML nu in de 'label' van de note
                    "hint": np.nan,
                    "guidance_hint": np.nan,
                    "relevant": "",
                    "appearance": ""
                })

                # Sluit de uitklapbare groep netjes af
                survey_list.append({
                    "type": "end group",
                    "name": "", 
                    "label": np.nan, 
                    "hint": np.nan, 
                    "guidance_hint": np.nan,
                    "relevant": "", 
                    "appearance": ""
                })

    # Sluit de groep (Pagina) af
    survey_list.append({
        "type": "end group",
        "name": "", "label": "", "relevant": "", "appearance": ""
    })

# Sluit de LSVI Groep af
survey_list.append({
    "type": "end group", "name": "", "label": "", 
    "relevant": "", "appearance": "", "default": "", "calculation": ""
})

df_survey_final = pd.DataFrame(survey_list)

Waarschuwing: Onbekend AnalyseVariabele type 'meting'. Fallback naar 'text' vraag.
Waarschuwing: Onbekend AnalyseVariabele type 'meting'. Fallback naar 'text' vraag.
Waarschuwing: Onbekend AnalyseVariabele type 'meting'. Fallback naar 'text' vraag.
Waarschuwing: Onbekend AnalyseVariabele type 'meting'. Fallback naar 'text' vraag.
Waarschuwing: Onbekend AnalyseVariabele type 'meting'. Fallback naar 'text' vraag.
Waarschuwing: Onbekend AnalyseVariabele type 'meting'. Fallback naar 'text' vraag.
Waarschuwing: Onbekend AnalyseVariabele type 'meting'. Fallback naar 'text' vraag.
Waarschuwing: Onbekend AnalyseVariabele type 'meting'. Fallback naar 'text' vraag.
Waarschuwing: Onbekend AnalyseVariabele type 'meting'. Fallback naar 'text' vraag.
Waarschuwing: Onbekend AnalyseVariabele type 'meting'. Fallback naar 'text' vraag.
Waarschuwing: Onbekend AnalyseVariabele type 'meting'. Fallback naar 'text' vraag.
Waarschuwing: Onbekend AnalyseVariabele type 'meting'. Fallback naar 'text' vraag.
Waar

### Add stresstest +1500 questions

In [75]:
# # Stresstest: add N additional questions to see if system (and feature layer in AGOL) has a limit

# n_additional_questions = 1200
# chunk_size = 400

# print(f"Generating {n_additional_questions} questions spread across multiple related tables...")

# for chunk_start in range(0, n_additional_questions, chunk_size):
#     chunk_num = (chunk_start // chunk_size) + 1

#     print('chunk num is ', chunk_num)
    
#     # 1. Start a new repeat (which creates a brand new database table)
#     survey_list.append({
#         "type": "begin repeat", 
#         "name": f"rpt_stress_table_{chunk_num}", 
#         "label": f"Stresstest Table {chunk_num}", 
#         "relevant": "", 
#         "appearance": "minimal",  # Hides the UI addition buttons
#         "repeat_count": "1"        # Seamlessly forces exactly 1 related record
#     })

#     print('chunk start is ', chunk_start)
    
#     # 2. Fill this specific table with questions up to the CHUNK_SIZE
#     chunk_end = min(chunk_start + chunk_size, n_additional_questions)
#     for i in range(chunk_start + 1, chunk_end + 1):
#         survey_list.append({
#             "type": "text", 
#             "name": f"stress_test_col_{i}", 
#             "label": f"Stresstest Vraag {i}", 
#             "relevant": "", 
#             "appearance": "minimal"
#         })
        
#     # 3. Close this repeat table
#     survey_list.append({
#         "type": "end repeat", "name": "", "label": "", "relevant": "", "appearance": ""
#     })

# df_survey_final = pd.DataFrame(survey_list)


In [101]:
df_survey_final.shape

(5207, 11)

### 2.4 Export to XLSX

In [119]:
import numpy as np
# Vervang alle lege strings in de hint (en eventueel andere) kolommen door echte NaN waarden
df_survey_final['hint'] = df_survey_final['hint'].replace("", np.nan)
# df_survey_final['guidance_hint'] = df_survey_final['guidance_hint'].replace("", np.nan)

# rename columns to add default language
df_survey_final = df_survey_final.rename(columns={'label': 'label::nl', 'hint': 'hint::nl', 'guidance_hint': 'guidance_hint::nl'})

print("Excel bestand genereren...")
output_file = '../output/Survey123_Volledige_Generatie_20260706.xlsx'
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_survey_final.to_excel(writer, sheet_name='survey', index=False)
    df_choices_final.to_excel(writer, sheet_name='choices', index=False)
    # Voeg een standaard settings tabblad toe
    pd.DataFrame([{"form_title": "Habitattype Survey", "form_id": "habitat_survey_v1", "default_language": "nl"}]).to_excel(writer, sheet_name='settings', index=False)

# # Overwrite Survey123 test app
# # output_file = r'C:\Users\joost_neujens\ArcGIS\My Survey Designs\LSVI App Test\LSVI App Test.xlsx' # if running local
# output_file = r"\\Client\C$\Users\joost_neujens\ArcGIS\My Survey Designs\LSVI App Test\LSVI App Test.xlsx"
# with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
#     df_survey_final.to_excel(writer, sheet_name='survey', index=False)
#     df_choices_final.to_excel(writer, sheet_name='choices', index=False)
#     # Voeg een standaard settings tabblad toe
#     settings_df.to_excel(writer, sheet_name='settings', index=False)

# print(f"✅ Survey123 XLSForm succesvol gegenereerd: '{output_file}'")

Excel bestand genereren...


In [117]:
df_test = df_survey_final.copy()
df_test['habtest'] = df_test['name'].str.extract(r'_(\d+)_[^_]+$')
df_test['habgroup'] = df_test['habtest'].astype(str).str[0]
df_test.groupby('habgroup').size().reset_index(name='row_count')

,habgroup,row_count
0,1,382
1,2,212
2,3,38
3,4,13
4,5,33
5,6,249
6,7,138
7,8,40
8,9,35


In [118]:
df_test.iloc[500:550]

,type,name,label,calculation,relevant,appearance,default,readonly,hint,guidance_hint,choice_filter,survey_part,habtest,habgroup
500,select_one LSVI,v451_kalkbedstro,Score:,NaN,,minimal w1,NaN,NaN,NaN,NaN,NaN,451,NaN,NaN
501,note,lbl_v451_duindravik,Duindravik,NaN,,w1,NaN,NaN,NaN,NaN,NaN,451,NaN,NaN
502,select_one LSVI,v451_duindravik,Score:,NaN,,minimal w1,NaN,NaN,NaN,NaN,NaN,451,NaN,NaN
503,note,lbl_v451_boompjesmos,Boompjesmos,NaN,,w1,NaN,NaN,NaN,NaN,NaN,451,NaN,NaN
504,select_one LSVI,v451_boompjesmos,Score:,NaN,,minimal w1,NaN,NaN,NaN,NaN,NaN,451,NaN,NaN
505,note,lbl_v451_geelzonneroosje,Geel zonneroosje,NaN,,w1,NaN,NaN,NaN,NaN,NaN,451,NaN,NaN
506,select_one LSVI,v451_geelzonneroosje,Score:,NaN,,minimal w1,NaN,NaN,NaN,NaN,NaN,451,NaN,NaN
507,end group,,,NaN,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
508,end group,,,NaN,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
509,begin group,grp_habitat_2150,Habitat 2150,NaN,"selected(string(${habitat_keuze}), '2150')",w1 compact field-list,NaN,NaN,2150: Geen omschrijving gevonden,NaN,NaN,NaN,NaN,NaN
